In [1]:
import sys
print(sys.version)
print(sys.executable)

import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

3.12.12 (main, Jan 13 2026, 17:36:25) [MSC v.1944 64 bit (AMD64)]
c:\pyproject1\gachikium\.venv\Scripts\python.exe
2.10.0+cu126
12.6
True


# 유저 기반 추천 시스템

## 1. 데이터 불러오기 

In [2]:
import pandas as pd

df = pd.read_csv('./data_hj/저출산 소개팅_설문조사_34건.csv')

In [19]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 34 entries, 0 to 33
Data columns (total 24 columns):
 #   Column                                                                                 Non-Null Count  Dtype
---  ------                                                                                 --------------  -----
 0   타임스탬프                                                                                  34 non-null     str  
 1   0. 당신의 성함                                                                              34 non-null     str  
 2   0. 당신의 이상형                                                                             34 non-null     str  
 3   1-1. 희망하는 자녀 수                                                                         34 non-null     str  
 4   1-2. 희망하는 자녀 구성                                                                        34 non-null     str  
 5   1-3. 자녀 갖고 싶은 시기                                                                       34 non-null     st

In [20]:
df.set_index('0. 당신의 성함', inplace=True)

In [21]:
df.drop(columns=['타임스탬프'], inplace=True, errors = 'ignore')

In [22]:
df.drop(columns=['Unnamed: 23'], inplace=True, errors = 'ignore')

In [24]:
print("현재 컬럼 목록:")
for i, col in enumerate(df.columns):
    print(f"{i}: {col}")

현재 컬럼 목록:
0: 0. 당신의 이상형
1: 1-1. 희망하는 자녀 수
2: 1-2. 희망하는 자녀 구성
3: 1-3. 자녀 갖고 싶은 시기
4: 1-4. 생물학적 출산이 어려움 발생 시 대안
5: "1. 자녀 계획 및 가족 구성 항목"에 대해 중요도 
6: 아이가 "오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?  
7: 평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.
8: 경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?
9: 재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?
10: 두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?
11: 한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다.
12: 맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?
13: 양가 어르신들이 본인의 가치관과 다른 육아 조언(예: "애를 너무 손타게 키운다", "사탕 좀 주면 어떠냐")을 하실 때 당신의 생각은?
14: 주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?
15: 아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?
16: 4-1. 자녀 교육비/양육비 지출 비중
17: 4-2. 육아 휴직, 양육 부담
18: "4. 경제적 지원 및 가사 분담"에 대해 중요도 
19: 5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? 
20: "5. 자녀 가치관"에 대해 중요도 


In [ ]:
#타임스탬프, 불필요 열 제거
#이름으로 인덱스 지정

df.to_csv('./data_hj/설문조사 1차 전처리.csv')  

In [25]:
df.head()

,0. 당신의 이상형,1-1. 희망하는 자녀 수,1-2. 희망하는 자녀 구성,1-3. 자녀 갖고 싶은 시기,1-4. 생물학적 출산이 어려움 발생 시 대안,"""1. 자녀 계획 및 가족 구성 항목""에 대해 중요도","아이가 ""오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?",평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.,"경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?","재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?",...,"한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다.",맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?,"양가 어르신들이 본인의 가치관과 다른 육아 조언(예: ""애를 너무 손타게 키운다"", ""사탕 좀 주면 어떠냐"")을 하실 때 당신의 생각은?","주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?",아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?,4-1. 자녀 교육비/양육비 지출 비중,"4-2. 육아 휴직, 양육 부담","""4. 경제적 지원 및 가사 분담""에 대해 중요도","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가?","""5. 자녀 가치관""에 대해 중요도"
0. 당신의 성함,,,,,,,,,,,,,,,,,,,,,
강현준,꽃돼지상,2명,"딸 1명, 아들 1명",결혼 즉시,의학적 도움 적극 시도;입양 고려,5,2,4,5,4,...,4,2,4,5,4,"노후 먼저, 남는 예산으로 지원","경제력 높은 사람 일하고, 한명은 전담 육아",1,"자신이 좋아하는 일, 행복",3
임경빈,꼬부기상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,4,3,4,4,2,...,4,4,4,4,3,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"회복탄력성, 생활력 강한 사람",3
이정미,곰상,2명,"딸 1명, 아들 1명",결혼 후 3~5년 이내,입양 고려,2,4,5,5,4,...,5,4,2,4,2,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",3,"회복탄력성, 생활력 강한 사람",4
쳌,고양이상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,1,3,4,5,4,...,5,4,2,5,2,노후보단 자녀 교육,"경제력 높은 사람 일하고, 한명은 전담 육아",2,"자신이 좋아하는 일, 행복",2
김용희,여우상,2명,"딸 1명, 아들 1명",결혼 후 1~2년 이내,의학적 도움 적극 시도,2,1,4,4,2,...,1,5,2,2,3,노후보단 자녀 교육,"맞벌이하면서 외부 도움(조부모, 시터)",2,"자신이 좋아하는 일, 행복",3


## 2. 데이터 EDA

## 3. 데이터 전처리

In [43]:
#객관식 데이터는 전체 선택지를 열로 만들어서 원핫 인코딩 처리, 다중 선택지는 멀티핫 인코딩 처리

import pandas as pd
from pandas.api.types import CategoricalDtype

#객관식 설문 조사 선택지 정의
# 모든 가능한 선택지를 정의 (예시 - 실제 설문지 내용으로 보강하세요)
all_categories = {
    '0. 당신의 이상형': ['강아지상', '고양이상', '사슴상', '토끼상', '꼬부기상','곰상','쿼카상','쥐상','도룡뇽상','여우상','말상','너구리상','꽃돼지상','햄스터상','늑대상','코알라상','오리상','원숭이','공룡상','알파카상, 라마상'],
    '1-1. 희망하는 자녀 수': ['1명', '2명', '3명', '그 이상'],
    '1-2. 희망하는 자녀 구성': ['오직 딸', '오직 아들', '딸 1명, 아들 1명'],
    '1-3. 자녀 갖고 싶은 시기': ['결혼 즉시', '결혼 후 1~2년 이내', '결혼 후 3~5년 이내', '경제적 안정 후'],
    '1-4. 생물학적 출산이 어려움 발생 시 대안': ['의학적 도움 적극 시도', '입양 고려', '무자녀'],
    '4-1. 자녀 교육비/양육비 지출 비중': ['노후보단 자녀 교육', '노후 먼저, 남는 예산으로 지원'],
    '4-2. 육아 휴직, 양육 부담': ['경제력 높은 사람 일하고, 한명은 전담 육아', '맞벌이하면서 외부 도움(조부모, 시터)'],
    '5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? ': ['경제적 성공, 사회적 지위', '도덕적, 타인 배려', '자신이 좋아하는 일, 행복', '회복탄력성, 생활력 강한 사람']
}

#객관식 답변 원핫 인코딩 진행

target_cols = [
    '0. 당신의 이상형', 
    '1-1. 희망하는 자녀 수', 
    '1-2. 희망하는 자녀 구성', 
    '1-3. 자녀 갖고 싶은 시기', 
    '1-4. 생물학적 출산이 어려움 발생 시 대안', 
    '4-1. 자녀 교육비/양육비 지출 비중 ', 
    '4-2. 육아 휴직, 양육 부담 ', 
    '5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? '
]

def encode_with_schema_v3(df, categories_dict):
    result_df = df.copy()
    
    for col, categories in categories_dict.items():
        if col not in result_df.columns:
            continue
            
        # 1. '기타' 처리 (단일 선택형)
        if col != '1-4. 생물학적 출산이 어려움 발생 시 대안':
            # 카테고리 리스트에 '기타'가 없다면 추가
            current_categories = categories.copy()
            if '기타' not in current_categories:
                current_categories.append('기타')
            
            # [수정 포인트] 리스트에 없는 값은 모두 '기타'라는 문자열로 변경
            result_df[col] = result_df[col].apply(lambda x: x if x in categories else '기타')
            
            # 카테고리 타입 적용
            dtype = CategoricalDtype(categories=current_categories)
            result_df[col] = result_df[col].astype(dtype)
            
            # 원-핫 인코딩
            dummies = pd.get_dummies(result_df[col], prefix=col)
            result_df = pd.concat([result_df, dummies], axis=1)
            result_df.drop(columns=[col], inplace=True)
            
        else:
            # 2. 다중 선택형(1-4번) 처리 (기존과 동일하게 유지하되 확실히 체크)
            for cat in categories:
                result_df[f"{col}_{cat}"] = result_df[col].astype(str).apply(
                    lambda x: 1 if cat in x else 0
                )
            
            # 정의되지 않은 답변이 하나라도 섞여 있다면 '기타'로 표시
            result_df[f"{col}_기타"] = result_df[col].apply(
                lambda x: 1 if any(item.strip() not in categories for item in str(x).split(';')) else 0
            )
            result_df.drop(columns=[col], inplace=True)
            
    return result_df

# 실행
df_encoded = encode_with_schema_v3(df, all_categories)

# 확인: '1-2. 희망하는 자녀 구성_기타' 열이 있는지 확인



In [44]:
target_check = [c for c in df_encoded.columns if '1-2. 희망하는 자녀 구성' in c]
print("생성된 컬럼들:", target_check)

생성된 컬럼들: ['1-2. 희망하는 자녀 구성_오직 딸', '1-2. 희망하는 자녀 구성_오직 아들', '1-2. 희망하는 자녀 구성_딸 1명, 아들 1명', '1-2. 희망하는 자녀 구성_기타']


In [45]:
df_encoded.to_csv('./data_hj/설문조사_2차_원핫인코딩_전처리.csv')

In [46]:
df_encoded.info()

<class 'pandas.DataFrame'>
Index: 34 entries, 강현준 to 유현희
Data columns (total 63 columns):
 #   Column                                                                                 Non-Null Count  Dtype
---  ------                                                                                 --------------  -----
 0   "1. 자녀 계획 및 가족 구성 항목"에 대해 중요도                                                          34 non-null     int64
 1   아이가 "오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?                                       34 non-null     int64
 2   평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.  34 non-null     int64
 3   경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?             34 non-null     int64
 4   재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?                         34 non-null     int64
 5   두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?                                            34 non-null     int6

In [47]:
#칼럼명 정제, 큰타옴표 불편함
df_encoded.columns = df_encoded.columns.str.replace('"', '').str.strip()


In [48]:
df_encoded.info()

<class 'pandas.DataFrame'>
Index: 34 entries, 강현준 to 유현희
Data columns (total 63 columns):
 #   Column                                                                                 Non-Null Count  Dtype
---  ------                                                                                 --------------  -----
 0   1. 자녀 계획 및 가족 구성 항목에 대해 중요도                                                            34 non-null     int64
 1   아이가 오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?                                        34 non-null     int64
 2   평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.  34 non-null     int64
 3   경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?             34 non-null     int64
 4   재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?                         34 non-null     int64
 5   두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?                                            34 non-null     int6

In [49]:
#가중치 설정 (중요도 설문 조사 값을 통해 각 섹션의 값 가중치 조절)

import numpy as np

def apply_weighted_logic(df):
    #섹션별 질문 컬럼 분류 

    w1_col = '1. 자녀 계획 및 가족 구성 항목에 대해 중요도'
    w4_col = '4. 경제적 지원 및 가사 분담에 대해 중요도'
    w5_col = '5. 자녀 가치관에 대해 중요도'

    section_1_cols = [c for c in df.columns if c.startswith(('1-1','1-2','1-3','1-4'))]
    section_4_cols = [c for c in df.columns if c.startswith(('4-1','4-2'))]
    section_5_cols = [c for c in df.columns if c.startswith(('5-1'))]

    # 중요도 가중치 계산 (1~5점을 0.2~1.0 가중치로 변환)
    w1 = (df[w1_col].values.astype(float) / 5.0).reshape(-1, 1)
    w4 = (df[w4_col].values.astype(float) / 5.0).reshape(-1, 1)
    w5 = (df[w5_col].values.astype(float) / 5.0).reshape(-1, 1)

    # 각 행별로 섹션 데이터에 가중치 곱하기
    df[section_1_cols] = df[section_1_cols].values * w1
    df[section_4_cols] = df[section_4_cols].values * w4
    df[section_5_cols] = df[section_5_cols].values * w5

    #중요도 칼럼은 제외시키기 
    df_weighted = df.drop(columns=[w1_col, w4_col, w5_col])

    return df_weighted

df_weighted = apply_weighted_logic(df_encoded)


In [51]:
df_weighted.to_csv('./data_hj/설문조사_3차_가중치 전처리.csv')

In [50]:
# 1. 대상 컬럼들에 어떤 타입의 데이터가 들어있는지 확인
print("--- 컬럼별 데이터 타입 확인 ---")
for col in [c for c in df_encoded.columns if '1-1' in c or '4-1' in c or '5-1' in c][:5]: # 일부만 확인
    sample_val = df_encoded[col].iloc[0]
    print(f"컬럼: {col} | 값: {sample_val} | 타입: {type(sample_val)}")

# 2. 혹시 숫자가 아닌 '객체(Object)'가 섞여 있는지 확인
print("\n--- 숫자가 아닌 값이 들어있는 컬럼 찾기 ---")
non_numeric_cols = df_encoded.select_dtypes(exclude=[np.number]).columns.tolist()
print(f"숫자가 아닌 컬럼 목록: {non_numeric_cols}")

--- 컬럼별 데이터 타입 확인 ---
컬럼: 1-1. 희망하는 자녀 수_1명 | 값: 0.0 | 타입: <class 'float'>
컬럼: 1-1. 희망하는 자녀 수_2명 | 값: 1.0 | 타입: <class 'float'>
컬럼: 1-1. 희망하는 자녀 수_3명 | 값: 0.0 | 타입: <class 'float'>
컬럼: 1-1. 희망하는 자녀 수_그 이상 | 값: 0.0 | 타입: <class 'float'>
컬럼: 1-1. 희망하는 자녀 수_기타 | 값: 0.0 | 타입: <class 'float'>

--- 숫자가 아닌 값이 들어있는 컬럼 찾기 ---
숫자가 아닌 컬럼 목록: ['0. 당신의 이상형_강아지상', '0. 당신의 이상형_고양이상', '0. 당신의 이상형_사슴상', '0. 당신의 이상형_토끼상', '0. 당신의 이상형_꼬부기상', '0. 당신의 이상형_곰상', '0. 당신의 이상형_쿼카상', '0. 당신의 이상형_쥐상', '0. 당신의 이상형_도룡뇽상', '0. 당신의 이상형_여우상', '0. 당신의 이상형_말상', '0. 당신의 이상형_너구리상', '0. 당신의 이상형_꽃돼지상', '0. 당신의 이상형_햄스터상', '0. 당신의 이상형_늑대상', '0. 당신의 이상형_코알라상', '0. 당신의 이상형_오리상', '0. 당신의 이상형_원숭이', '0. 당신의 이상형_공룡상', '0. 당신의 이상형_알파카상, 라마상', '0. 당신의 이상형_기타', '1-1. 희망하는 자녀 수_1명', '1-1. 희망하는 자녀 수_2명', '1-1. 희망하는 자녀 수_3명', '1-1. 희망하는 자녀 수_그 이상', '1-1. 희망하는 자녀 수_기타', '1-2. 희망하는 자녀 구성_오직 딸', '1-2. 희망하는 자녀 구성_오직 아들', '1-2. 희망하는 자녀 구성_딸 1명, 아들 1명', '1-2. 희망하는 자녀 구성_기타', '1-3. 자녀 갖고 싶은 시기_결혼 즉시', '1-3. 자녀 갖고 싶은 시기_결혼 후 1~2년 이내', '1-3. 자녀 갖

In [55]:
# 시나리오 질문 기반 파생 변수 생성

def create_upbringing_traits(df):
    # 0. 컬럼명 정제 (공백 및 따옴표 제거)
    df.columns = df.columns.str.replace('"', '').str.strip()
    
    # 1. 섹션별 질문 매핑 (질문 문구의 핵심 키워드로 매칭)
    trait_mapping = {
        '양육_스타일_SF': [
            '아이가 오늘만 양치 안하고 그냥 자면 안돼요? 라고 칭얼거릴 때 어떻게 하시겠습니까?',
            '평소 밤 9시에 자기로 약속했습니다. 그런데 오늘 아이가 읽고 싶어 하던 동화책 시리즈의 마지막 권을 다 읽고 싶다며 30분만 더 시간을 달라고 합니다.'
        ],
        '교육_우선순위_AH': [
            '경쟁 상황에서의 태도 아이가 운동 경기나 대회에서 아쉽게 2등을 했습니다. 아이는 충분히 잘했다고 기뻐하는데, 당신의 마음속 생각은?',
            '재능 발견과 교육 아이가 특정 분야(예: 피아노, 운동)에 천재적인 재능을 보입니다. 이때 당신의 교육 방향은?'
        ],
        '공동_양육_EL': [
            '두 사람의 훈육 방식이 부딪힐 때, 누구의 의견을 따라야 한다고 생각하시나요?',
            '한 명은 퇴근 후 아이와 놀아주고, 한 명은 밀린 집안일을 해야 하는 상황입니다.'
        ],
        '확장_가족_BT': [
            '맞벌이 상황 등에서 조부모님이 아이를 봐주겠다고 제안하신다면?',
            '양가 어르신들이 본인의 가치관과 다른 육아 조언(예: 애를 너무 손타게 키운다, 사탕 좀 주면 어떠냐)을 하실 때 당신의 생각은?'
        ],
        '계획_대응_PG': [
            '주말에 아이와 동물원에 가기로 했는데, 아침에 일어나니 갑자기 비가 옵니다. 이때 당신의 반응은?',
            '아이의 교육 자금이나 미래 리스크를 대비하는 당신의 생각은?'
        ]
    }

    # 2. 각 섹션별 합산 점수 계산
    for trait_name, questions in trait_mapping.items():
        # 질문이 데이터에 있는지 확인 후 합산 (1~5점 질문 2개이므로 합은 2~10점)
        available_cols = [q for q in questions if q in df.columns]
        if len(available_cols) == 2:
            df[trait_name] = df[available_cols[0]].astype(int) + df[available_cols[1]].astype(int)
        else:
            print(f"경고: {trait_name}를 위한 질문 컬럼을 찾을 수 없습니다.")

    # 3. 기존 개별 시나리오 질문 컬럼은 삭제 (이제 '성향 점수'가 그 역할을 대신함)
    all_scenario_cols = [q for sublist in trait_mapping.values() for q in sublist if q in df.columns]
    df_traits = df.drop(columns=all_scenario_cols)
    
    return df_traits

# 실행
df_with_traits = create_upbringing_traits(df_weighted)

# 결과 확인
print("새로 생성된 성향 컬럼들:")
print(df_with_traits[['양육_스타일_SF', '교육_우선순위_AH', '공동_양육_EL', '확장_가족_BT', '계획_대응_PG']].head()) 

새로 생성된 성향 컬럼들:
           양육_스타일_SF  교육_우선순위_AH  공동_양육_EL  확장_가족_BT  계획_대응_PG
0. 당신의 성함                                                     
강현준                6           9         8         6         9
임경빈                7           6         8         8         7
이정미                9           9         9         6         6
쳌                  7           9         8         6         7
김용희                5           6         5         7         5


In [56]:
df_with_traits.info()

<class 'pandas.DataFrame'>
Index: 34 entries, 강현준 to 유현희
Data columns (total 55 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   0. 당신의 이상형_강아지상                                 34 non-null     bool   
 1   0. 당신의 이상형_고양이상                                 34 non-null     bool   
 2   0. 당신의 이상형_사슴상                                  34 non-null     bool   
 3   0. 당신의 이상형_토끼상                                  34 non-null     bool   
 4   0. 당신의 이상형_꼬부기상                                 34 non-null     bool   
 5   0. 당신의 이상형_곰상                                   34 non-null     bool   
 6   0. 당신의 이상형_쿼카상                                  34 non-null     bool   
 7   0. 당신의 이상형_쥐상                                   34 non-null     bool   
 8   0. 당신의 이상형_도룡뇽상                                 34 non-null     bool   
 9   0. 당신의 이상형_여우상                                  34 non-nul

In [57]:
df_with_traits.to_csv('./data_hj/설문조사_4차_시나리오 정리.csv')

In [58]:
def scale_traits_fixed(df):
    trait_cols = ['양육_스타일_SF', '교육_우선순위_AH', '공동_양육_EL', '확장_가족_BT', '계획_대응_PG']
    
    for col in trait_cols:
        if col in df.columns:
            # 2~10점 범위를 0~1로 변환
            # (값 - 최소값) / (최대값 - 최소값)
            df[col] = (df[col] - 2) / (10 - 2)
            
            # 혹시라도 범위를 벗어나는 값이 생길 경우를 대비해 0~1 사이로 제한(Clip)
            df[col] = df[col].clip(0, 1)
            
    return df

# 실행
df_traits_scaled = scale_traits_fixed(df_with_traits)

# 확인
print("성향 점수 스케일링 확인 (8점이었던 경우 0.75가 되어야 함):")
print(df_traits_scaled[['양육_스타일_SF']].head())

성향 점수 스케일링 확인 (8점이었던 경우 0.75가 되어야 함):
           양육_스타일_SF
0. 당신의 성함           
강현준            0.500
임경빈            0.625
이정미            0.875
쳌              0.625
김용희            0.375


In [60]:
df_traits_scaled.to_csv('./data_hj/설문조사_5차_시나리오 정규화.csv')

In [11]:
import pandas as pd
df = pd.read_csv('./data_hj/설문조사_5차_시나리오 정규화.csv')

In [12]:
df.set_index('0. 당신의 성함', inplace=True)
df.head(2)

,0. 당신의 이상형_강아지상,0. 당신의 이상형_고양이상,0. 당신의 이상형_사슴상,0. 당신의 이상형_토끼상,0. 당신의 이상형_꼬부기상,0. 당신의 이상형_곰상,0. 당신의 이상형_쿼카상,0. 당신의 이상형_쥐상,0. 당신의 이상형_도룡뇽상,0. 당신의 이상형_여우상,...,"5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? _경제적 성공, 사회적 지위","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? _도덕적, 타인 배려","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? _자신이 좋아하는 일, 행복","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? _회복탄력성, 생활력 강한 사람","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? _기타",양육_스타일_SF,교육_우선순위_AH,공동_양육_EL,계획_대응_PG,확장_가족_BT
0. 당신의 성함,,,,,,,,,,,,,,,,,,,,,
강현준,False,False,False,False,False,False,False,False,False,False,...,0.0,0.0,0.6,0.0,0.0,0.500,0.875,0.75,0.875,0.50
임경빈,False,False,False,False,True,False,False,False,False,False,...,0.0,0.0,0.0,0.6,0.0,0.625,0.500,0.75,0.625,0.75


In [13]:
ideltype_cols = [c for c in df.columns if c.startswith('0.')]

df = df.drop(columns=ideltype_cols)
df.head(2)

,1-1. 희망하는 자녀 수_1명,1-1. 희망하는 자녀 수_2명,1-1. 희망하는 자녀 수_3명,1-1. 희망하는 자녀 수_그 이상,1-1. 희망하는 자녀 수_기타,1-2. 희망하는 자녀 구성_오직 딸,1-2. 희망하는 자녀 구성_오직 아들,"1-2. 희망하는 자녀 구성_딸 1명, 아들 1명",1-2. 희망하는 자녀 구성_기타,1-3. 자녀 갖고 싶은 시기_결혼 즉시,...,"5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? _경제적 성공, 사회적 지위","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? _도덕적, 타인 배려","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? _자신이 좋아하는 일, 행복","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? _회복탄력성, 생활력 강한 사람","5-1. 자녀 가치관, 어떤 사람이 되길 바라는가? _기타",양육_스타일_SF,교육_우선순위_AH,공동_양육_EL,계획_대응_PG,확장_가족_BT
0. 당신의 성함,,,,,,,,,,,,,,,,,,,,,
강현준,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,...,0.0,0.0,0.6,0.0,0.0,0.500,0.875,0.75,0.875,0.50
임경빈,0.0,0.8,0.0,0.0,0.0,0.0,0.0,0.8,0.0,0.0,...,0.0,0.0,0.0,0.6,0.0,0.625,0.500,0.75,0.625,0.75


## 4. 유사도 확인

In [18]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

val_cols = [c for c in df.columns if any(q in c for q in ['1-1', '1-2', '1-3', '1-4', '4-1', '4-2', '5-1'])]
trait_cols = ['양육_스타일_SF', '교육_우선순위_AH', '공동_양육_EL', '확장_가족_BT', '계획_대응_PG']

#유사도 계산
val_sim_matrix = cosine_similarity(df[val_cols], df[val_cols])
val_sim_matrix

array([[1.        , 0.59071735, 0.36380344, ..., 0.61579354, 0.22140372,
        0.12249899],
       [0.59071735, 1.        , 0.53425846, ..., 0.83992105, 0.22951012,
        0.42857143],
       [0.36380344, 0.53425846, 1.        , ..., 0.37416574, 0.10954451,
        0.64649763],
       ...,
       [0.61579354, 0.83992105, 0.37416574, ..., 1.        , 0.58554004,
        0.2159797 ],
       [0.22140372, 0.22951012, 0.10954451, ..., 0.58554004, 1.        ,
        0.07377111],
       [0.12249899, 0.42857143, 0.64649763, ..., 0.2159797 , 0.07377111,
        1.        ]], shape=(34, 34))

In [15]:
val_sim_df = pd.DataFrame(
    val_sim_matrix,
    index = df.index,
    columns= df.index
)


In [17]:
val_sim_df.to_csv('./data_hj/코사인유사도_테이블_val.csv')

## 5. 유사도 높은 사람 추천

In [29]:
#가치관 점수는 유사하고, 사니라오 점수는 유사하거나 반대되는 사람 찾기
def get_smart_recommendations(user_name, top_n=5):
    if user_name not in df.index:
        return '사용자를 찾을 수 없습니다.'
    
    #내 MBTI 가져오기
    my_traits = df.loc[user_name, trait_cols]

    #모든 유저 리스트 생성
    recommendations = []
    for other_user in df.index:
        if user_name == other_user:
            continue

        #가치관 유사도 점수
        val_score = val_sim_df.loc[user_name, other_user]

        #시나리오 성향 계산 
        other_traits = df.loc[other_user, trait_cols]
        trait_diff = np.abs(my_traits - other_traits).mean()

        #유저별 가치관 유사도, 시나리오 성향 차이 리스트로 정리
        recommendations.append({
            'name' : other_user,
            'val_score' : val_score,
            'trait_diff' : trait_diff,
            'trait_type' : '보완형(Flipped)' if trait_diff > 0.3 else '동질형(Similar)'
        })

    recom_df = pd.DataFrame(recommendations).sort_values(by='val_score', ascending=False)
    print(recom_df)
    print(f'{user_name}님의 맞춤 리스트')
    print('-'*50)

    top_similar = recom_df[recom_df['trait_type'] == '동질형(Similar)'].head(2)
    top_flipped = recom_df[recom_df['trait_type'] == '보완형(Flipped)'].head(2)

    final_recom = pd.concat([top_similar, top_flipped])

    for i, row in final_recom.iterrows():
        print(f"[{row['trait_type']}] {row['name']} | 가치관 유사도: {row['val_score']*100:.1f}% | 성향 차이: {row['trait_diff']:.2f}")
        
    return final_recom


In [40]:
get_smart_recommendations('강현준')

           name  val_score  trait_diff    trait_type
13      언리얼/이동진   0.730178       0.175  동질형(Similar)
26          하준우   0.625799       0.225  동질형(Similar)
30          이은수   0.615794       0.175  동질형(Similar)
0           임경빈   0.590717       0.200  동질형(Similar)
3           김용희   0.582154       0.300  동질형(Similar)
23           ㅁㅎ   0.533507       0.200  동질형(Similar)
2             쳌   0.493058       0.075  동질형(Similar)
9           이주헌   0.454754       0.375  보완형(Flipped)
10          박재혁   0.453743       0.150  동질형(Similar)
11          송형진   0.447811       0.175  동질형(Similar)
29          최서진   0.428746       0.400  보완형(Flipped)
6           임태나   0.425436       0.075  동질형(Similar)
20          최미정   0.425343       0.250  동질형(Similar)
27           샌새   0.387816       0.275  동질형(Similar)
14          누구게   0.365089       0.250  동질형(Similar)
1           이정미   0.363803       0.175  동질형(Similar)
15           연근   0.356512       0.075  동질형(Similar)
16          박소현   0.353553       0.225  동질형(Si

,name,val_score,trait_diff,trait_type
13,언리얼/이동진,0.730178,0.175,동질형(Similar)
26,하준우,0.625799,0.225,동질형(Similar)
9,이주헌,0.454754,0.375,보완형(Flipped)
29,최서진,0.428746,0.400,보완형(Flipped)


In [24]:
def check_user_match_details(target_user):
    if target_user not in df.index:
        return "사용자를 찾을 수 없습니다."
    
    # 1. 내 성향 데이터 가져오기
    my_traits = df.loc[target_user, trait_cols]
    
    details = []
    for other_user in df.index:
        if target_user == other_user: continue
        
        # 가치관 유사도
        val_score = val_sim_df.loc[target_user, other_user]
        
        # 시나리오 성향 차이 (절대값의 평균)
        other_traits = df.loc[other_user, trait_cols]
        diffs = np.abs(my_traits - other_traits)
        mean_diff = diffs.mean()
        
        details.append({
            '상대방': other_user,
            '가치관 유사도': f"{val_score*100:.1f}%",
            '성향 차이(평균)': round(mean_diff, 3),
            '최대 차이 항목': diffs.idxmax(), # 어떤 성향에서 가장 많이 차이나는지
            '유형': '보완형(Flipped)' if mean_diff > 0.3 else '동질형(Similar)' # 기준을 0.3으로 낮춰서 확인
        })
    
    # 결과 출력
    result_df = pd.DataFrame(details).sort_values(by='성향 차이(평균)', ascending=False)
    print(f"--- [{target_user}] 님 기준 모든 유저 매칭 상세 정보 ---")
    return result_df

# 실행 (본인의 이름을 넣으세요)
check_user_match_details('강현준')

--- [강현준] 님 기준 모든 유저 매칭 상세 정보 ---


,상대방,가치관 유사도,성향 차이(평균),최대 차이 항목,유형
29,최서진,42.9%,0.400,계획_대응_PG,보완형(Flipped)
9,이주헌,45.5%,0.375,양육_스타일_SF,보완형(Flipped)
25,임세희,4.2%,0.350,계획_대응_PG,보완형(Flipped)
3,김용희,58.2%,0.300,계획_대응_PG,동질형(Similar)
27,샌새,38.8%,0.275,공동_양육_EL,동질형(Similar)
20,최미정,42.5%,0.250,공동_양육_EL,동질형(Similar)
18,안보영,0.0%,0.250,계획_대응_PG,동질형(Similar)
14,누구게,36.5%,0.250,계획_대응_PG,동질형(Similar)
31,민채영,22.1%,0.250,양육_스타일_SF,동질형(Similar)
26,하준우,62.6%,0.225,교육_우선순위_AH,동질형(Similar)


## 6. 상호 보완적 사람 추천

## 7. MBTI 유형 뽑기 (본인)

## 8. MBTI 궁합 유형 추천(상대)